# 19b — Compare Target Scoring V1 vs V2

This notebook compares the original V1 scoring model against the calibrated V2 scoring model.

It answers:

- Which wards entered or left the top lists?
- Which councils moved up or down?
- How much did the rescaled demographic component change the watchlist?
- Are the biggest movements politically sensible or artefacts?


## 19b.1 Project paths and settings


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
V1_OUTPUT_DIR = PROCESSED_DIR / "target_model_v1" / "outputs"
V2_OUTPUT_DIR = PROCESSED_DIR / "target_model_v2" / "outputs"
V2_WATCHLIST_DIR = PROCESSED_DIR / "target_model_v2" / "watchlists"
COMPARE_DIR = PROCESSED_DIR / "target_model_v2" / "v1_v2_comparison"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_REGION = "North West"
TOP_N = 50

print("Project:", PROJECT_DIR)
print("Compare output folder:", COMPARE_DIR)


Project: c:\Users\keena\Documents\Electoral_Tribes
Compare output folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\v1_v2_comparison


## 19b.2 Load V1 and V2 component files


In [2]:
v1_path = V1_OUTPUT_DIR / "target_score_components_ward25_all_available_v1.csv"
v2_path = V2_OUTPUT_DIR / "target_score_components_ward25_all_available_v2.csv"

if not v1_path.exists():
    raise FileNotFoundError(f"Missing V1 score components: {v1_path}")
if not v2_path.exists():
    raise FileNotFoundError(f"Missing V2 score components: {v2_path}. Run 18b first.")

v1 = pd.read_csv(v1_path, low_memory=False)
v2 = pd.read_csv(v2_path, low_memory=False)

print("V1 rows:", len(v1))
print("V2 rows:", len(v2))
print("V1 columns:", v1.columns.tolist())
print("V2 columns:", v2.columns.tolist())


V1 rows: 7572
V2 rows: 7572
V1 columns: ['LAD25CD', 'LAD25NM', 'WD25CD', 'WD25NM', 'analysis_region', 'dominant_cluster_name', 'second_cluster_name', 'initial_watchlist_score', 'initial_watchlist_percentile', 'review_band', 'demographic_relevance_score', 'electoral_opportunity_score', 'political_openness_score', 'data_confidence_score', 'margin_competitiveness_score', 'top_party_dominance_inverse_score', 'valid_vote_threshold_score', 'electoral_fragmentation_score', 'non_main_party_score', 'effective_parties_score', 'lab_con_inverse_score', 'data_confidence_note', 'boundary_caveat', 'county_election_caveat']
V2 columns: ['LAD25CD', 'LAD25NM', 'WD25CD', 'WD25NM', 'analysis_region', 'dominant_cluster_name', 'second_cluster_name', 'initial_watchlist_score', 'initial_watchlist_percentile', 'review_band', 'review_band_clean', 'demographic_relevance_score', 'electoral_opportunity_score', 'political_openness_score', 'data_confidence_score', 'margin_competitiveness_score', 'top_party_dominance

## 19b.3 Merge scores and calculate movements


In [3]:
key_cols = ["LAD25CD", "WD25CD"]

v1_keep = [c for c in v1.columns if c in key_cols or c in [
    "LAD25NM", "WD25NM", "analysis_region", "dominant_cluster_name", "second_cluster_name",
    "initial_watchlist_score", "initial_watchlist_percentile", "review_band",
    "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "data_confidence_score",
    "boundary_caveat", "county_election_caveat", "data_confidence_note"
]]

v2_keep = [c for c in v2.columns if c in key_cols or c in [
    "LAD25NM", "WD25NM", "analysis_region", "dominant_cluster_name", "second_cluster_name",
    "initial_watchlist_score", "initial_watchlist_percentile", "review_band", "review_band_clean",
    "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "data_confidence_score",
    "breakthrough_complacency_score", "major_party_safe_seat_score", "low_turnout_apathy_score",
    "latest_election_top_party_bucket", "latest_election_margin_pct_allocated", "latest_election_top_party_share", "latest_election_turnout_proxy",
    "boundary_caveat", "county_election_caveat", "data_confidence_note", "has_major_caveat"
]]

merged = v1[v1_keep].merge(
    v2[v2_keep],
    on=key_cols,
    how="inner",
    suffixes=("_v1", "_v2"),
    validate="one_to_one"
)

merged["score_change_v2_minus_v1"] = merged["initial_watchlist_score_v2"] - merged["initial_watchlist_score_v1"]
merged["demographic_score_change"] = merged["demographic_relevance_score_v2"] - merged["demographic_relevance_score_v1"]
merged["rank_v1"] = merged["initial_watchlist_score_v1"].rank(ascending=False, method="min")
merged["rank_v2"] = merged["initial_watchlist_score_v2"].rank(ascending=False, method="min")
merged["rank_change_v2_minus_v1"] = merged["rank_v1"] - merged["rank_v2"]

merged.to_csv(COMPARE_DIR / "v1_v2_score_comparison_all_available_v2.csv", index=False)

print("Merged rows:", len(merged))
display(merged[["score_change_v2_minus_v1", "demographic_score_change", "rank_change_v2_minus_v1"]].describe())


Merged rows: 7572


,score_change_v2_minus_v1,demographic_score_change,rank_change_v2_minus_v1
count,7572.000000,7572.000000,7572.000000
mean,9.180526,29.945107,0.000000
std,6.498007,12.096446,972.758746
min,-9.542224,-8.662108,-3733.000000
25%,6.013531,26.965264,-500.000000
50%,9.856951,31.988951,-52.000000
75%,12.636480,37.421953,350.000000
max,46.424920,55.000000,6267.000000


## 19b.4 Scope comparison function


In [4]:
def safe_slug(name):
    return re.sub(r"[^a-zA-Z0-9]+", "_", str(name).lower()).strip("_")


def compare_scope(frame, scope_name, region_name=None, top_n=50):
    if region_name is not None:
        region_col = "analysis_region_v2" if "analysis_region_v2" in frame.columns else "analysis_region_v1"
        scope = frame[frame[region_col].eq(region_name)].copy()
    else:
        scope = frame.copy()

    slug = safe_slug(scope_name)

    top_v1 = scope.sort_values("initial_watchlist_score_v1", ascending=False).head(top_n).copy()
    top_v2 = scope.sort_values("initial_watchlist_score_v2", ascending=False).head(top_n).copy()

    top_v1_keys = set(zip(top_v1["LAD25CD"], top_v1["WD25CD"]))
    top_v2_keys = set(zip(top_v2["LAD25CD"], top_v2["WD25CD"]))

    overlap_keys = top_v1_keys & top_v2_keys
    entered_keys = top_v2_keys - top_v1_keys
    exited_keys = top_v1_keys - top_v2_keys

    def flag_membership(row):
        key = (row["LAD25CD"], row["WD25CD"])
        if key in overlap_keys:
            return "in_top_both"
        if key in entered_keys:
            return "entered_top_v2"
        if key in exited_keys:
            return "left_top_v2"
        return "outside_top_both"

    scope["top_list_movement"] = scope.apply(flag_membership, axis=1)

    overlap_summary = pd.DataFrame([{
        "scope": scope_name,
        "top_n": top_n,
        "top_v1_count": len(top_v1),
        "top_v2_count": len(top_v2),
        "overlap_count": len(overlap_keys),
        "entered_v2_count": len(entered_keys),
        "left_v2_count": len(exited_keys),
        "overlap_pct_of_top_n": len(overlap_keys) / top_n * 100 if top_n else np.nan,
    }])

    # Save movement review.
    movement = scope[scope["top_list_movement"].ne("outside_top_both")].sort_values(
        ["top_list_movement", "initial_watchlist_score_v2"],
        ascending=[True, False]
    )
    movement.to_csv(COMPARE_DIR / f"v1_v2_top{top_n}_movement_{slug}_v2.csv", index=False)
    overlap_summary.to_csv(COMPARE_DIR / f"v1_v2_top{top_n}_overlap_summary_{slug}_v2.csv", index=False)

    # Biggest changes.
    biggest_up = scope.sort_values("score_change_v2_minus_v1", ascending=False).head(100)
    biggest_down = scope.sort_values("score_change_v2_minus_v1", ascending=True).head(100)
    biggest_up.to_csv(COMPARE_DIR / f"v1_v2_biggest_score_increases_{slug}_v2.csv", index=False)
    biggest_down.to_csv(COMPARE_DIR / f"v1_v2_biggest_score_decreases_{slug}_v2.csv", index=False)

    # Council summary.
    lad_name_col = "LAD25NM_v2" if "LAD25NM_v2" in scope.columns else "LAD25NM_v1"
    region_col = "analysis_region_v2" if "analysis_region_v2" in scope.columns else "analysis_region_v1"
    council = (
        scope.groupby(["LAD25CD", lad_name_col, region_col], dropna=False, as_index=False)
        .agg(
            ward_count=("WD25CD", "nunique"),
            mean_score_v1=("initial_watchlist_score_v1", "mean"),
            mean_score_v2=("initial_watchlist_score_v2", "mean"),
            max_score_v1=("initial_watchlist_score_v1", "max"),
            max_score_v2=("initial_watchlist_score_v2", "max"),
            review_a_v1=("review_band_v1", lambda x: (x == "Review A").sum()),
            review_a_v2=("review_band_v2", lambda x: (x == "Review A").sum()),
            clean_review_ab_v2=("review_band_clean", lambda x: x.isin(["Review A Clean", "Review B Clean"]).sum() if x.notna().any() else 0),
            caveated_review_ab_v2=("review_band_clean", lambda x: x.isin(["Review A Caveated", "Review B Caveated"]).sum() if x.notna().any() else 0),
            mean_breakthrough_v2=("breakthrough_complacency_score", "mean"),
        )
    )
    council["mean_score_change"] = council["mean_score_v2"] - council["mean_score_v1"]
    council = council.sort_values(["clean_review_ab_v2", "mean_score_v2"], ascending=[False, False])
    council.to_csv(COMPARE_DIR / f"v1_v2_council_summary_comparison_{slug}_v2.csv", index=False)

    print(f"\nScope: {scope_name}")
    display(overlap_summary)
    print("Top V2 entries:")
    display(top_v2.head(20))
    return overlap_summary, movement, council

# Default North West comparison.
nw_overlap, nw_movement, nw_council = compare_scope(merged, "north_west", region_name=DEFAULT_REGION, top_n=TOP_N)

# All available comparison.
all_overlap, all_movement, all_council = compare_scope(merged, "all_available", region_name=None, top_n=TOP_N)



Scope: north_west


,scope,top_n,top_v1_count,top_v2_count,overlap_count,entered_v2_count,left_v2_count,overlap_pct_of_top_n
0,north_west,50,50,50,35,15,15,70.0


Top V2 entries:


,LAD25CD,LAD25NM_v1,WD25CD,WD25NM_v1,analysis_region_v1,dominant_cluster_name_v1,second_cluster_name_v1,initial_watchlist_score_v1,initial_watchlist_percentile_v1,review_band_v1,demographic_relevance_score_v1,electoral_opportunity_score_v1,political_openness_score_v1,data_confidence_score_v1,data_confidence_note_v1,boundary_caveat_v1,county_election_caveat_v1,LAD25NM_v2,WD25NM_v2,analysis_region_v2,dominant_cluster_name_v2,second_cluster_name_v2,initial_watchlist_score_v2,initial_watchlist_percentile_v2,review_band_v2,review_band_clean,demographic_relevance_score_v2,electoral_opportunity_score_v2,political_openness_score_v2,data_confidence_score_v2,breakthrough_complacency_score,major_party_safe_seat_score,low_turnout_apathy_score,latest_election_top_party_bucket,latest_election_margin_pct_allocated,latest_election_top_party_share,latest_election_turnout_proxy,data_confidence_note_v2,boundary_caveat_v2,county_election_caveat_v2,has_major_caveat,score_change_v2_minus_v1,demographic_score_change,rank_v1,rank_v2,rank_change_v2_minus_v1
3389,E07000117,Burnley,E05005152,Brunshaw,North West,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,61.603515,99.550977,Review A,39.766922,77.948165,61.202571,90.0,County election caveat.,NaN,Latest election layer may include 2025 county ...,Burnley,Brunshaw,North West,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,79.763938,99.788695,Review A,Review A Caveated,88.226373,75.345019,73.124807,80,0.000000,0.000000,32.951804,independent,0.074756,0.378522,0.301717,County election caveat.,NaN,Latest election layer may include 2025 county ...,True,18.160423,48.459451,35.0,17.0,18.0
3458,E07000121,Lancaster,E05014894,Heysham North,North West,Settled Working Families / Skilled Trades Suburbs,Post-Industrial Estates / Deprived Working Com...,61.230889,99.418912,Review A,36.768382,84.495015,52.053803,100.0,No major caveat.,NaN,NaN,Lancaster,Heysham North,North West,Settled Working Families / Skilled Trades Suburbs,Post-Industrial Estates / Deprived Working Com...,78.611324,99.617010,Review A,Review A Clean,81.165382,83.218786,62.951216,95,54.253246,31.070196,43.650099,lab,0.056387,0.310702,0.253575,No major caveat.,NaN,NaN,False,17.380435,44.397001,45.0,30.0,15.0
180,E06000009,Blackpool,E05015206,Waterloo,North West,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,58.023453,96.275753,Review A,37.088564,82.791170,40.820420,100.0,No major caveat.,NaN,NaN,Blackpool,Waterloo,North West,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,75.953840,98.930269,Review A,Review A Clean,82.480342,81.775542,52.212231,95,50.343514,32.155074,24.388404,lab,0.031357,0.321551,0.340252,No major caveat.,NaN,NaN,False,17.930386,45.391778,283.0,82.0,201.0
5318,E08000004,Oldham,E05014652,Failsworth East,North West,Settled Working Families / Skilled Trades Suburbs,Rooted Older Homeowners,58.064163,96.315372,Review A,31.005592,81.521234,51.023343,100.0,No major caveat.,NaN,NaN,Oldham,Failsworth East,North West,Settled Working Families / Skilled Trades Suburbs,Rooted Older Homeowners,75.034628,98.335975,Review A,Review A Clean,70.661396,81.132744,63.853265,100,49.322090,33.091897,37.901469,lab,0.008601,0.330919,0.279443,No major caveat.,NaN,NaN,False,16.970465,39.655804,280.0,127.0,153.0
5248,E08000001,Bolton,E05014823,Farnworth South,North West,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,57.888177,96.077655,Review A,39.801052,71.606147,49.903860,100.0,No major caveat.,NaN,NaN,Bolton,Farnworth South,North West,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,74.769921,98.217116,Review A,Review A Clean,87.221810,66.303463,57.404996,100,0.000000,0.000000,47.598051,other,0.098498,0.430654,0.235809,No major caveat.,NaN,NaN,False,16


Scope: all_available


,scope,top_n,top_v1_count,top_v2_count,overlap_count,entered_v2_count,left_v2_count,overlap_pct_of_top_n
0,all_available,50,50,50,23,27,27,46.0


Top V2 entries:


,LAD25CD,LAD25NM_v1,WD25CD,WD25NM_v1,analysis_region_v1,dominant_cluster_name_v1,second_cluster_name_v1,initial_watchlist_score_v1,initial_watchlist_percentile_v1,review_band_v1,demographic_relevance_score_v1,electoral_opportunity_score_v1,political_openness_score_v1,data_confidence_score_v1,data_confidence_note_v1,boundary_caveat_v1,county_election_caveat_v1,LAD25NM_v2,WD25NM_v2,analysis_region_v2,dominant_cluster_name_v2,second_cluster_name_v2,initial_watchlist_score_v2,initial_watchlist_percentile_v2,review_band_v2,review_band_clean,demographic_relevance_score_v2,electoral_opportunity_score_v2,political_openness_score_v2,data_confidence_score_v2,breakthrough_complacency_score,major_party_safe_seat_score,low_turnout_apathy_score,latest_election_top_party_bucket,latest_election_margin_pct_allocated,latest_election_top_party_share,latest_election_turnout_proxy,data_confidence_note_v2,boundary_caveat_v2,county_election_caveat_v2,has_major_caveat,score_change_v2_minus_v1,demographic_score_change,rank_v1,rank_v2,rank_change_v2_minus_v1
2779,E07000081,Gloucester,E05010954,Coney Hill,South West,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,63.221176,99.894348,Review A,40.083799,81.497696,58.970152,100.0,No major caveat.,NaN,NaN,Gloucester,Coney Hill,South West,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,81.990184,100.000000,Review A,Review A Clean,87.709497,80.958068,68.017759,100,0.000000,0.000000,44.938522,independent,0.073149,0.362562,0.247777,No major caveat.,NaN,NaN,False,18.769008,47.625698,9.0,1.0,8.0
3787,E07000137,East Lindsey,E05009903,Trinity,East Midlands,Post-Industrial Estates / Deprived Working Com...,Stable Suburban Professionals,62.140596,99.749076,Review A,34.259709,84.380539,59.342144,100.0,No major caveat.,NaN,NaN,East Lindsey,Trinity,East Midlands,Post-Industrial Estates / Deprived Working Com...,Stable Suburban Professionals,81.407491,99.986793,Review A,Review A Clean,80.976942,85.122377,70.115392,100,0.000000,0.000000,25.844076,reform_ukip_brexit,0.020603,0.342899,0.333702,No major caveat.,NaN,NaN,False,19.266895,46.717233,20.0,2.0,18.0
6821,W06000001,Isle of Anglesey,W05001503,Tref Cybi,Wales,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,60.870898,99.220814,Review A,42.495413,82.068978,49.507242,90.0,No major caveat.,NaN,NaN,Isle of Anglesey,Tref Cybi,Wales,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,81.143971,99.973587,Review A,Review A Clean,94.495413,82.176051,57.671047,90,0.000000,0.000000,23.003242,independent,0.039234,0.352153,0.346485,No major caveat.,NaN,NaN,False,20.273073,52.000000,60.0,3.0,57.0
1477,E06000057,Northumberland,E05016102,Cramlington East & Double Row,North East,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,61.911834,99.683043,Review A,38.321834,85.795167,51.042566,100.0,No major caveat.,NaN,NaN,Northumberland,Cramlington East & Double Row,North East,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,81.099785,99.960380,Review A,Review A Clean,85.846185,87.052781,59.751144,100,0.000000,0.000000,33.232887,independent,0.005310,0.307080,0.300452,No major caveat.,NaN,NaN,False,19.187951,47.524351,25.0,4.0,21.0
2951,E07000090,Havant,E05015588,Leigh Park Central & West Leigh,South East,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,65.894989,100.000000,Review A,42.225868,85.038770,62.417216,100.0,No major caveat.,NaN,NaN,Havant,Leigh Park Central & West Leigh,South East,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,80.997105,99.947174,Review A,Review A Clean,93.386826,81.764684,55.129244,100,43.280034,16.929439,0.000000,lab,0.058155,0.169294,0.450145,No major caveat.,Na

## 19b.5 Review V2 watchlist files

This section loads the V2 watchlists created by 18b and displays the clean, caveated, demographic-build and breakthrough lists for the North West.


In [ ]:
watchlist_files = {
    "clean": V2_WATCHLIST_DIR / "clean_watchlist_review_ab_north_west_v2.csv",
    "caveated": V2_WATCHLIST_DIR / "caveated_watchlist_review_ab_north_west_v2.csv",
    "demographic_build": V2_WATCHLIST_DIR / "demographic_build_watchlist_north_west_v2.csv",
    "breakthrough": V2_WATCHLIST_DIR / "breakthrough_complacency_watchlist_north_west_v2.csv",
}

for label, path in watchlist_files.items():
    print("\n", label, path)
    if path.exists():
        temp = pd.read_csv(path, low_memory=False)
        print("Rows:", len(temp))
        cols = [
            "LAD25NM", "WD25NM", "dominant_cluster_name", "latest_election_top_party_bucket",
            "initial_watchlist_score", "review_band_clean", "demographic_relevance_score", "electoral_opportunity_score",
            "political_openness_score", "breakthrough_complacency_score", "data_confidence_score",
            "data_confidence_note"
        ]
        cols = [c for c in cols if c in temp.columns]
        display(temp[cols].head(25))
    else:
        print("Missing file")



 clean c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\clean_watchlist_review_ab_north_west_v2.csv
Rows: 49


,LAD25NM,WD25NM,dominant_cluster_name,latest_election_top_party_bucket,initial_watchlist_score,review_band_clean,demographic_relevance_score,electoral_opportunity_score,political_openness_score,breakthrough_complacency_score,data_confidence_score,data_confidence_note
0,Lancaster,Heysham North,Settled Working Families / Skilled Trades Suburbs,lab,78.611324,Review A Clean,81.165382,83.218786,62.951216,54.253246,95,No major caveat.
1,Blackpool,Waterloo,Post-Industrial Estates / Deprived Working Com...,lab,75.953840,Review A Clean,82.480342,81.775542,52.212231,50.343514,95,No major caveat.
2,Oldham,Failsworth East,Settled Working Families / Skilled Trades Suburbs,lab,75.034628,Review A Clean,70.661396,81.132744,63.853265,49.322090,100,No major caveat.
3,Bolton,Farnworth South,Post-Industrial Estates / Deprived Working Com...,other,74.769921,Review A Clean,87.221810,66.303463,57.404996,0.000000,100,No major caveat.
4,Blackpool,Bloomfield,Post-Industrial Estates / Deprived Working Com...,lab,73.799538,Review A Clean,95.363090,69.017176,40.869214,68.561254,95,No major caveat.
5,Warrington,Orford,Post-Industrial Estates / Deprived Working Com...,lab,73.616050,Review A Clean,77.618709,76.091622,54.488060,39.107102,100,No major caveat.
6,Cumberland,Moss Bay and Moorclose,Post-Industrial Estates / Deprived Working Com...,independent,73.273791,Review A Clean,90.153490,73.886166,42.216878,0.000000,90,No major caveat.
7,Wirral,Rock Ferry,Post-Industrial Estates / Deprived Working Com...,lab,72.442747,Review A Clean,90.512520,57.065929,56.574345,56.704397,95,No major caveat.
8,Wigan,Ince,Post-Industrial Estates / Deprived Working Com...,independent,72.437902,Review A Clean,89.509410,68.318230,42.456559,0.000000,100,No major caveat.
9,Liverpool,Old Swan East,Settled Working Families / Skilled Trades Suburbs,lab,72.235706,Review A Clean,82.660851,71.255548,49.710977,57.459162,95,No major caveat.



 caveated c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\caveated_watchlist_review_ab_north_west_v2.csv
Rows: 24


,LAD25NM,WD25NM,dominant_cluster_name,latest_election_top_party_bucket,initial_watchlist_score,review_band_clean,demographic_relevance_score,electoral_opportunity_score,political_openness_score,breakthrough_complacency_score,data_confidence_score,data_confidence_note
0,Burnley,Brunshaw,Post-Industrial Estates / Deprived Working Com...,independent,79.763938,Review A Caveated,88.226373,75.345019,73.124807,0.00000,80,County election caveat.
1,Burnley,Trinity,Post-Industrial Estates / Deprived Working Com...,reform_ukip_brexit,72.851244,Review A Caveated,90.492668,60.562736,60.039959,0.00000,80,County election caveat.
2,Lancaster,West End,Settled Working Families / Skilled Trades Suburbs,reform_ukip_brexit,72.560025,Review A Caveated,83.402382,69.769666,57.753167,0.00000,80,County election caveat.
3,Lancaster,Scale Hall,Post-Industrial Estates / Deprived Working Com...,reform_ukip_brexit,72.508986,Review A Caveated,82.333800,64.844702,64.954984,0.00000,80,County election caveat.
4,Lancaster,Carnforth & Millhead,Settled Working Families / Skilled Trades Suburbs,reform_ukip_brexit,70.266574,Review B Caveated,64.996690,83.169377,58.267675,0.00000,80,County election caveat.
5,Ribble Valley,Littlemoor,Settled Working Families / Skilled Trades Suburbs,reform_ukip_brexit,70.107035,Review B Caveated,65.143258,79.910545,61.334926,0.00000,80,County election caveat.
6,Chorley,Chorley East,Settled Working Families / Skilled Trades Suburbs,reform_ukip_brexit,69.619259,Review B Caveated,77.365064,69.155315,55.179565,0.00000,80,County election caveat.
7,South Ribble,Seven Stars,Post-Industrial Estates / Deprived Working Com...,reform_ukip_brexit,69.348558,Review B Caveated,78.621847,69.850069,51.503565,0.00000,80,County election caveat.
8,Burnley,Gawthorpe,Post-Industrial Estates / Deprived Working Com...,reform_ukip_brexit,69.260146,Review B Caveated,80.596488,68.898936,49.526775,0.00000,80,County election caveat.
9,Preston,Brookfield,Post-Industrial Estates / Deprived Working Com...,reform_ukip_brexit,68.979629,Review B Caveated,85.196493,58.002120,55.040882,0.00000,80,County election caveat.



 demographic_build c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\demographic_build_watchlist_north_west_v2.csv
Rows: 150


,LAD25NM,WD25NM,dominant_cluster_name,latest_election_top_party_bucket,initial_watchlist_score,review_band_clean,demographic_relevance_score,electoral_opportunity_score,political_openness_score,breakthrough_complacency_score,data_confidence_score,data_confidence_note
0,Liverpool,Kirkdale East,Post-Industrial Estates / Deprived Working Com...,lab,58.472062,Review D Clean,100.000000,28.249663,21.988652,83.094891,95,No major caveat.
1,Liverpool,Clubmoor East,Post-Industrial Estates / Deprived Working Com...,lab,61.330804,Review D Clean,97.695445,30.745787,33.654649,78.501615,95,No major caveat.
2,Halton,Central & West Bank,Post-Industrial Estates / Deprived Working Com...,lab,61.010410,Review D Clean,97.691927,31.708969,29.222181,82.006640,100,No major caveat.
3,Liverpool,Belle Vale,Post-Industrial Estates / Deprived Working Com...,lab,57.943176,Review D Clean,97.100256,19.540350,34.383925,73.396199,95,No major caveat.
4,Knowsley,Northwood,Post-Industrial Estates / Deprived Working Com...,lab,59.423648,Review D Clean,97.067251,24.891357,31.930810,80.885370,100,No major caveat.
5,Liverpool,Speke,Post-Industrial Estates / Deprived Working Com...,lab,61.856288,Review C Clean,96.858648,26.154201,42.438005,72.665152,95,No major caveat.
6,St. Helens,Parr,Post-Industrial Estates / Deprived Working Com...,lab,51.515155,Review D Clean,95.844724,20.289017,11.531185,81.068807,90,No major caveat.
7,Salford,Little Hulton,Post-Industrial Estates / Deprived Working Com...,lab,56.019164,Review D Clean,94.325556,24.182320,23.002093,76.923404,100,No major caveat.
8,Cheshire West and Chester,Wolverham,Post-Industrial Estates / Deprived Working Com...,lab,54.041958,Review D Clean,94.107296,26.304432,14.852301,77.808666,95,No major caveat.
9,St. Helens,Peasley Cross & Fingerpost,Post-Industrial Estates / Deprived Working Com...,lab,60.544503,Review D Clean,93.686263,38.176909,29.204954,73.133279,90,No major caveat.



 breakthrough c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v2\watchlists\breakthrough_complacency_watchlist_north_west_v2.csv
Rows: 128


,LAD25NM,WD25NM,dominant_cluster_name,latest_election_top_party_bucket,initial_watchlist_score,review_band_clean,demographic_relevance_score,electoral_opportunity_score,political_openness_score,breakthrough_complacency_score,data_confidence_score,data_confidence_note
0,Liverpool,Kirkdale East,Post-Industrial Estates / Deprived Working Com...,lab,58.472062,Review D Clean,100.000000,28.249663,21.988652,83.094891,95,No major caveat.
1,Knowsley,Stockbridge,Post-Industrial Estates / Deprived Working Com...,lab,54.100813,Review D Clean,93.135791,19.398858,22.734514,82.928227,100,No major caveat.
2,Halton,Central & West Bank,Post-Industrial Estates / Deprived Working Com...,lab,61.010410,Review D Clean,97.691927,31.708969,29.222181,82.006640,100,No major caveat.
3,Sefton,Linacre,Post-Industrial Estates / Deprived Working Com...,lab,48.598049,Review D Caveated,92.282737,14.525449,13.765826,81.816200,85,Boundary caveat.
4,West Lancashire,Skelmersdale South,Post-Industrial Estates / Deprived Working Com...,lab,48.834406,Review D Clean,90.742660,15.678556,9.483633,81.670211,100,No major caveat.
5,St. Helens,Parr,Post-Industrial Estates / Deprived Working Com...,lab,51.515155,Review D Clean,95.844724,20.289017,11.531185,81.068807,90,No major caveat.
6,Knowsley,Northwood,Post-Industrial Estates / Deprived Working Com...,lab,59.423648,Review D Clean,97.067251,24.891357,31.930810,80.885370,100,No major caveat.
7,Knowsley,Cherryfield,Post-Industrial Estates / Deprived Working Com...,lab,51.084428,Review D Clean,87.588208,16.863841,21.477612,79.178212,100,No major caveat.
8,Liverpool,Clubmoor East,Post-Industrial Estates / Deprived Working Com...,lab,61.330804,Review D Clean,97.695445,30.745787,33.654649,78.501615,95,No major caveat.
9,West Lancashire,Tanhouse & Skelmersdale Town Centre,Post-Industrial Estates / Deprived Working Com...,lab,54.291337,Review D Clean,91.765648,24.090131,19.785286,78.135140,100,No major caveat.


: 